In [ ]:
# Install required packages first:
# pip install pandas requests python-dotenv openpyxl

import requests
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import datetime

# Load your BEARER_TOKEN from .env
load_dotenv()
BEARER_TOKEN = os.getenv("BEARER_TOKEN")

if not BEARER_TOKEN:
    raise ValueError("Bearer token not found. Add BEARER_TOKEN to your .env file.")

# Headers for API request
headers = {
    "Authorization": f"Bearer {BEARER_TOKEN}"
}

# Load usernames from CSV
# Make sure your CSV has a column named 'username'
accounts_df = pd.read_csv(
    r"c:\Users\abidh\OneDrive\Documents\accounts_list.csv",
    encoding="latin1"  # fixes UnicodeDecodeError
)
usernames = accounts_df["username"].tolist()

# Limit to 100 usernames (Twitter API limit per request)
if len(usernames) > 100:
    print("Only the first 100 usernames will be processed.")
    usernames = usernames[:100]

# Join usernames into a comma-separated string
username_string = ",".join(usernames)

# Twitter API endpoint to get users by username
url = "https://api.twitter.com/2/users/by"

params = {
    "usernames": username_string,
    "user.fields": "id,name,username,public_metrics,verified,created_at,description"
}

# Make the request
try:
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()  # Raise HTTPError if not 200

    response_json = response.json()
    
    if "data" not in response_json:
        print("No 'data' field in response. Full response:")
        print(response_json)
        exit()

    users = response_json["data"]

except requests.exceptions.HTTPError as err:
    print("HTTP Error:", err)
    print("Response:", response.text)
    exit()
except Exception as e:
    print("Error fetching data:", e)
    exit()

# Process user data
data = []
for user in users:
    metrics = user.get("public_metrics", {})
    data.append({
        "Username": user.get("username"),
        "Name": user.get("name"),
        "Followers": metrics.get("followers_count"),
        "Following": metrics.get("following_count"),
        "Tweet Count": metrics.get("tweet_count"),
        "Listed Count": metrics.get("listed_count"),
        "Verified": user.get("verified"),
        "Created At": user.get("created_at"),
        "Bio": user.get("description"),
        "Data Retrieved On": datetime.now().strftime("%Y-%m-%d")
    })

# Convert to DataFrame
df = pd.DataFrame(data)

# Save to CSV and Excel
df.to_csv("twitter_100_accounts_data.csv", index=False)
df.to_excel("twitter_100_accounts_data.xlsx", index=False)

print("✅ Data successfully saved! Rows fetched:", len(df))

Error: {
  "title": "Unauthorized",
  "type": "about:blank",
  "status": 401,
  "detail": "Unauthorized"
}


KeyError: 'data'

: 